In [2]:
import pandas as pd

# ۱. مسیر فایل 
file_path = r"C:\Users\Lenovo\Downloads\20683008\GIMENDO_v1_Metadata.xlsx"

# ۲. خواندن فایل و استفاده از ردیف دوم (header=1) به عنوان نام ستون‌ها
df = pd.read_excel(file_path, header=1)

# چاپ نام ستون‌ها برای اطمینان از اینکه همه چیز درست خوانده شده
print("نام ستون‌های پیدا شده:")
print(df.columns.tolist())
print("-" * 50)

# ۳. فیلتر کردن ردیف‌ها
df = df[df['Media Type'].str.contains('Image', na=False)]
df = df.dropna(subset=['Histological Subtype'])

# ۴. ساده‌سازی کلاس‌ها
def clean_label(text):
    text = str(text).lower()
    if 'complete + incomplete' in text or 'both' in text:
        return 'Mixed'
    elif 'incomplete' in text:
        return 'Incomplete'
    elif 'complete' in text:
        return 'Complete'
    elif 'negative' in text:
        return 'Negative'
    return 'Other'

df['Clean_Label'] = df['Histological Subtype'].apply(clean_label)

# ۵. گزارش‌گیری
print("تعداد کل تصاویر معتبر (واجد شرایط):", len(df))
print("-" * 40)
print("تعداد نمونه‌ها در هر کلاس (توزیع داده‌ها):")
print(df['Clean_Label'].value_counts())

نام ستون‌های پیدا شده:
['Case Code', 'Media Type', 'Age', 'Sex', 'LBC', 'MTB', 'WOS', 'TVF', 'IRV', 'MLE', 'TM', 'Histological Subtype', 'Atrophy', 'Dysplasia', 'OLGIM', 'OLGA', 'H. pylori (Pathology)', 'Gastropathy', 'Processor', 'Scope']
--------------------------------------------------
تعداد کل تصاویر معتبر (واجد شرایط): 21
----------------------------------------
تعداد نمونه‌ها در هر کلاس (توزیع داده‌ها):
Clean_Label
Incomplete    7
Complete      7
Mixed         3
Other         3
Negative      1
Name: count, dtype: int64


In [3]:
import os
import shutil
import re

# ۱. مسیر پوشه دانلودها (کمی عقب‌تر از پوشه اصلی جستجو می‌کنیم تا چیزی جا نماند)
source_base_dir = r"C:\Users\Lenovo\Downloads\20683008"

# ۲. مسیر مقصد برای ساخت پوشه‌های تمیز
target_base_dir = r"C:\Users\Lenovo\Downloads\20683008\Organized_Dataset"

# ایجاد پوشه مقصد
os.makedirs(target_base_dir, exist_ok=True)

image_count = 0
video_count = 0
patients_found = set()

print("در حال استخراج قطعی فایل‌ها...")

for root, dirs, files in os.walk(source_base_dir):
    # جلوگیری از جستجو داخل پوشه مقصدی که خودمان داریم می‌سازیم (جلوگیری از لوپ بی‌نهایت)
    if "Organized_Dataset" in root:
        continue
        
    for file in files:
        is_image = file.lower().endswith(('.png', '.jpg', '.jpeg'))
        is_video = file.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))
        
        if is_image or is_video:
            # جستجوی قطعی: پیدا کردن الگوی "26" به علاوه دو رقم (مثل 2601 یا 2615)
            # این الگو حتی در اسم‌هایی مثل img2601 هم کار می‌کند
            match = re.search(r'(26\d{2})', root)
            if not match:
                match = re.search(r'(26\d{2})', file)
                
            if match:
                case_code = match.group(1)
                patients_found.add(case_code)
                
                # ساخت پوشه‌ها
                patient_folder = os.path.join(target_base_dir, f"Patient_{case_code}")
                media_folder_name = "Images" if is_image else "Videos"
                final_dest_folder = os.path.join(patient_folder, media_folder_name)
                
                os.makedirs(final_dest_folder, exist_ok=True)
                
                # کپی فایل
                src_path = os.path.join(root, file)
                dst_path = os.path.join(final_dest_folder, file)
                
                if not os.path.exists(dst_path):
                    shutil.copy2(src_path, dst_path)
                    
                    if is_image:
                        image_count += 1
                    elif is_video:
                        video_count += 1

print("-" * 50)
print("عملیات با موفقیت به پایان رسید!")
print(f"تعداد کل بیماران استخراج شده: {len(patients_found)} نفر")
print(f"تعداد کل عکس‌های مرتب‌شده: {image_count} عدد")
print(f"تعداد کل ویدیوهای مرتب‌شده: {video_count} عدد")

در حال استخراج قطعی فایل‌ها...
--------------------------------------------------
عملیات با موفقیت به پایان رسید!
تعداد کل بیماران استخراج شده: 23 نفر
تعداد کل عکس‌های مرتب‌شده: 99 عدد
تعداد کل ویدیوهای مرتب‌شده: 37 عدد
